# Setup & Daten Laden

In [ ]:
import os
import json
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Memory Growth für TF
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)

from data import load_data, prepare_splits, NUM_CLASSES, LABEL_DICT
from models import VARIANT_V5_DEEPER_DENSE, VARIANT_V3_DOUBLE_CONV_WITH_BN, VARIANT_V2_MINIMAL_WITH_BN_GAP, build_model

DATA_PATH = "./../data/Galaxy10_DECals.h5"
BATCH_SIZE = 32
TEMPERATURE = 5.0
ALPHA = 0.1 

REPORTS_DIR = Path("./../reports/knowledge_distillation")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Lade Bilder und erstelle identische Splits...")
images_raw, labels = load_data(DATA_PATH)
split_folds, X_test, y_test = prepare_splits(
    labels, use_kfold=False, val_size=0.2, test_size=0.2, random_state=42
)
train_idx, val_idx = split_folds[0]

print("Lade Teacher-Logits von Festplatte...")
train_logits = np.load("teacher_train_logits.npy")
val_logits = np.load("teacher_val_logits.npy")

assert len(train_idx) == len(train_logits), "Fehler: Train-Logits passen nicht zum Daten-Split!"
assert len(val_idx) == len(val_logits), "Fehler: Val-Logits passen nicht zum Daten-Split!"

def prepare_tf_dataset(indices, teacher_logits, batch_size, is_training=True):
    x_data = images_raw[indices]
    y_data = labels[indices]
    dataset = tf.data.Dataset.from_tensor_slices((x_data, y_data, teacher_logits))
    if is_training:
        dataset = dataset.shuffle(buffer_size=1024)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_dataset = prepare_tf_dataset(train_idx, train_logits, BATCH_SIZE, is_training=True)
val_dataset = prepare_tf_dataset(val_idx, val_logits, BATCH_SIZE, is_training=False)

# Distiller Klasse & Training

In [ ]:
class Distiller(keras.Model):
    def __init__(self, student, temperature=3.0, alpha=0.1):
        super().__init__()
        self.student = student
        self.temperature = temperature
        self.alpha = alpha
        self.student_loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        self.distillation_loss_fn = keras.losses.KLDivergence()

    def compile(self, optimizer, metrics):
        super().compile(optimizer=optimizer, metrics=metrics)

    def train_step(self, data):
        x, y, teacher_logits = data
        with tf.GradientTape() as tape:
            student_logits = self.student(x, training=True)
            student_loss = self.student_loss_fn(y, student_logits)

            # Beide Logits werden durch Temperatur und Softmax skaliert
            soft_teacher_probs = tf.nn.softmax(teacher_logits / self.temperature, axis=1)
            soft_student_probs = tf.nn.softmax(student_logits / self.temperature, axis=1)

            distillation_loss = self.distillation_loss_fn(soft_teacher_probs, soft_student_probs)
            loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss

        gradients = tape.gradient(loss, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.student.trainable_variables))
        
        # Metriken aktualisieren
        self.compiled_metrics.update_state(y, student_logits)
        results = {m.name: m.result() for m in self.metrics}
        results.update({"stud_loss": student_loss, "dist_loss": distillation_loss})
        return results
        
    def test_step(self, data):
        x, y, _ = data
        student_logits = self.student(x, training=False)
        student_loss = self.student_loss_fn(y, student_logits)
        self.compiled_metrics.update_state(y, student_logits)
        results = {m.name: m.result() for m in self.metrics}
        results.update({"val_loss": student_loss})
        return results

# 2. Schleife über unsere besten kleinen Architekturen!
candidates = {
    "V5_KD (685k)": VARIANT_V5_DEEPER_DENSE,
    "V3_KD (298k)": VARIANT_V3_DOUBLE_CONV_WITH_BN,
    "V2_KD (103k)": VARIANT_V2_MINIMAL_WITH_BN_GAP
}

trained_students = {}

print("Starte Knowledge Distillation Loop...")

for name, architecture in candidates.items():
    print(f"\n{'='*50}\nTrainiere Student: {name}\n{'='*50}")
    
    # Modell initialisieren
    student_model = build_model(architecture, learning_rate=1e-4, output_logits=True)
    
    # Distiller aufbauen
    distiller = Distiller(student=student_model, temperature=TEMPERATURE, alpha=ALPHA)
    distiller.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
    )
    
    # Trainieren
    distiller.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=30, # Evtl. anpassen (30 reicht meist für kleinere Modelle mit Teacher)
        callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)],
        verbose=1
    )
    
    # Speichern für die spätere Evaluation
    trained_students[name] = student_model

print("\nAlle Studenten wurden erfolgreich trainiert!")

# Evaluation & Plots

In [ ]:
X_test_tf = tf.data.Dataset.from_tensor_slices(images_raw[X_test]).batch(BATCH_SIZE)

models_data = {
    # --- Station 1 Modelle (aus architecture_ranking.csv) ---
    "Station 1 (v7_five_conv_blocks)": {"params": 2.492, "acc": 82.36},
    "Station 1 (9_conv_layer_cnn)": {"params": 2.492, "acc": 82.33},
    "Station 1 (v9_with_512_dense)": {"params": 2.755, "acc": 82.27},
    "Station 1 (v12_with_dense_bn)": {"params": 2.493, "acc": 82.19},
    "Station 1 (v5_deeper_dense)": {"params": 0.685, "acc": 80.67},
    "Station 1 (v6_with_dropout_03)": {"params": 0.685, "acc": 78.69},
    "Station 1 (v8_five_conv_dropout)": {"params": 2.492, "acc": 77.93},
    "Station 1 (v11_best_cnn_deeper_dense)": {"params": 2.755, "acc": 76.94},
    "Station 1 (v4_deeper_filters)": {"params": 0.619, "acc": 75.14},
    "Station 1 (v10_best_cnn_lite)": {"params": 1.244, "acc": 74.04},
    "Station 1 (v3_double_conv_bn)": {"params": 0.298, "acc": 70.69},
    "Station 1 (minimal_cnn)": {"params": 33.648, "acc": 58.40},
    "Station 1 (v1_minimal_with_bn)": {"params": 33.649, "acc": 45.89},
    "Station 1 (v2_minimal_bn_gap)": {"params": 0.103, "acc": 45.24},

    # --- Station 2 Modelle (aus Tabelle/Bild) ---
    "Station 2 (ResNet-50)": {"params": 24.89, "acc": 38.64},
    "Station 2 (ResNet-152)": {"params": 59.68, "acc": 43.55},
    "Station 2 (Inception V3)": {"params": 22.54, "acc": 61.22},
    "Station 2 (Zoobot ConvNeXt-Nano frozen)": {"params": 15.11, "acc": 88.33},
    "Station 2 (Zoobot ConvNeXt-Nano finetuned)": {"params": 15.11, "acc": 89.12}
}

# Füge die destillierten Modelle zur Statistik hinzu
for name, model in trained_students.items():
    y_pred_probs = model.predict(X_test_tf, verbose=1)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    acc = float(np.mean(y_pred == y_test) * 100) 
    params_mio = float(model.count_params() / 1_000_000)
    
    models_data[name] = {"params": params_mio, "acc": acc}
    print(f"{name} -> Test Accuracy nach KD: {acc:.2f}%")

# ==========================================
# 1. Plot: Parameter vs Accuracy
# ==========================================
fig, ax = plt.subplots(figsize=(12, 7))

for name, info in models_data.items():
    # Unterscheide alte Modelle vs. neue KD-Modelle farblich
    if "KD" in name:
        color, marker, size = "red", "*", 250 
    elif "Teacher" in name:
        color, marker, size = "green", "^", 150
    else:
        color, marker, size = "blue", "o", 100
        
    ax.scatter(info["params"], info["acc"], label=name, color=color, marker=marker, s=size)
    
    # Text-Labels leicht versetzt zeichnen
    ax.text(info["params"] * 1.05, info["acc"], name, fontsize=10, 
            verticalalignment='bottom' if "Teacher" in name else 'top')

ax.set_xscale("log")
ax.set_xlabel("Parameteranzahl (Millionen, logarithmische Skala)")
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Wunder der Knowledge Distillation: Modellgröße vs. Performance")
ax.grid(True, which="both", ls="--", alpha=0.5)

# X-Ticks schöner formatieren (0.1, 1, 10)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: '{:g}'.format(y)))

# --- Speichern des Scatterplots ---
plot_path = REPORTS_DIR / "parameter_vs_accuracy.png"
fig.savefig(plot_path, dpi=200, bbox_inches="tight")
print(f"\n[INFO] Scatterplot gespeichert unter: {plot_path}")
plt.show()
plt.close(fig)

# ==========================================
# 2. Confusion Matrix & Modell-Export
# ==========================================
# Finde das beste KD-Modell anhand der Test Accuracy
best_kd_name = max({k: v for k, v in models_data.items() if "KD" in k}, key=lambda k: models_data[k]['acc'])
best_kd_model = trained_students[best_kd_name]

y_pred_probs = best_kd_model.predict(X_test_tf, verbose=0)
best_y_pred = np.argmax(y_pred_probs, axis=1)
cm = confusion_matrix(y_test, best_y_pred)
class_names = [LABEL_DICT[i] for i in range(NUM_CLASSES)]

fig_cm, ax_cm = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax_cm)
ax_cm.set_title(f"Confusion Matrix - Bestes KD Modell ({best_kd_name} mit {models_data[best_kd_name]['acc']:.2f}%)")
ax_cm.set_ylabel("Ground Truth")
ax_cm.set_xlabel("Prediction")
plt.xticks(rotation=45, ha="right")

# --- Speichern der Confusion Matrix ---
cm_path = REPORTS_DIR / "best_model_confusion_matrix.png"
fig_cm.savefig(cm_path, dpi=200, bbox_inches="tight")
print(f"[INFO] Confusion Matrix gespeichert unter: {cm_path}")
plt.show()
plt.close(fig_cm)

# ==========================================
# 3. JSON & Weights Export
# ==========================================
json_path = REPORTS_DIR / "distillation_results.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(models_data, f, indent=4)
print(f"[INFO] Ergebnisse als JSON gespeichert unter: {json_path}")

model_path = REPORTS_DIR / "best_student_model.keras"
best_kd_model.save(model_path)
print(f"[INFO] Bestes Studenten-Modell gespeichert unter: {model_path}")